# EduHarness — Stage 2 API Steps: Prepare / Review

Despite the historical filename, local rendering is now the default. Use this notebook only for API-backed `2-prepare` and `2-review`.

Default loop:

1. Set `MODE = "prepare"` and run this notebook.
2. Sync `runs/<id>` to local and run CLI stage `2-render` (no API key).
3. Sync the run back to Drive, set `MODE = "review"`, and rerun.
4. If review fails, repeat from step 1; prepare will target failed scenes using saved feedback.


In [ ]:
from __future__ import annotations
import subprocess, sys
from pathlib import Path

MODE = "prepare"
assert MODE in {"prepare", "review"}

from eduharness.colab_session import ensure_api_key, load_session
session = load_session()
ensure_api_key()

REPO, RUN_ROOT = Path(session["REPO"]), Path(session["RUN_DIR"])

if MODE == "review":
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])

from eduharness.config import Config
from eduharness.pipeline import load_request, stage2_prepare, stage2_review

REQUEST_PATH = Path(session["REQUEST"])
request = load_request(REQUEST_PATH)
cfg = Config.from_env(request, run_dir=RUN_ROOT / request.request_id)
assert cfg.blueprint_path.is_file(), "Run Stage 1 first"

if MODE == "prepare":
    prepared = stage2_prepare(cfg)
    print(f"Prepared {len(prepared)} scenes in {cfg.run_dir}")
    print(
        "python -m eduharness --request-json examples/sample_request.json "
        f"--run-dir runs/{request.request_id} --stage 2-render"
    )
else:
    assert cfg.rendered_path.is_file(), "Sync the locally rendered run back to Drive first"
    rendered = stage2_review(cfg)
    failed = [r.scene_id for r in rendered if not r.review_passed]
    for r in rendered:
        print(f"{r.scene_id:12} {r.scene_type.value:12} pass={r.review_passed} src={r.src}")
    if failed:
        print("Review failed:", failed)
    else:
        print("All scenes passed. Sync locally, then run stage 3.")
